In [ ]:
import json
from pathlib import Path
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

RELEVES_ROOT = "releves"
SITE = "chronovet"  # on peut changer pour un autre site

conn = psycopg2.connect(
    dbname="vetprice",
    user="postgres",
    password="Mkilo1990",
    host="localhost",
    port=5432
)
cur = conn.cursor()

def cle_produit(row):
    ean = row.get("ean")
    if isinstance(ean, str) and ean.strip():
        return ean
    return row["url"]

def charger_site(site):
    runs = sorted((Path(RELEVES_ROOT) / site).iterdir())
    dfs = []
    for run_dir in runs:
        f = run_dir / "products.jsonl"
        if not f.exists():
            continue
        df = pd.read_json(f, lines=True, dtype=False)
        df["site"] = site
        df["cle"] = df.apply(cle_produit, axis=1)
        df["valid_from"] = pd.to_datetime(df["scraped_at"]).dt.date
        dfs.append(df)
    if not dfs:
        return None
    return pd.concat(dfs, ignore_index=True)

df_all = charger_site(SITE)

df_hist = df_all[[
    "site", "cle", "ean", "url", "name", "brand",
    "price", "in_stock", "valid_from"
]].copy()

df_hist = df_hist.drop_duplicates(subset=["site", "cle", "valid_from"], keep="first")

df_hist["valid_to"] = None
df_hist["is_current"] = True

rows = [
    (
        r["site"],
        r["cle"],
        r.get("ean"),
        r.get("url"),
        r.get("name"),
        r.get("brand"),
        r.get("price"),
        r.get("in_stock"),
        r["valid_from"],
        r["valid_to"],
        r["is_current"],
    )
    for _, r in df_hist.iterrows()
]

sql = """
INSERT INTO produit_historise (
    site, cle, ean, url, name, brand,
    price, in_stock, valid_from, valid_to, is_current
)
VALUES %s
ON CONFLICT DO NOTHING;
"""

execute_values(cur, sql, rows)
conn.commit()
print(f"{len(rows)} lignes insérées dans produit_historise pour le site {SITE}.")
